# Обработка признаков

В этом домашнем задании вы будете решать задачу предсказания стоимости автомобилей по их различным характеристикам.

In [1]:
import pandas as pd
import numpy as np

RANDOM_STATE = 42

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/evgpat/edu_stepik_practical_ml/main/datasets/cars_prices.csv", decimal='.')

In [3]:
df.head()

,symboling,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,...,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
0,3,?,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495
1,3,?,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500
2,1,?,alfa-romero,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500
3,2,164,audi,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950
4,2,164,audi,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450


### Описание некоторых признаков

`symboling` - rating corresponds to the degree to which the auto is more risky than its price indicates (+3 more risk and -3 is pretty safe)  
`make` - car types (i.e. car brand)  
`fuel-type` - types of fuel (gas or diesel)  
`aspiration` - engine aspiration (standard or turbo)  
`num-of-doors` - numbers of doors (two or four)  
`body-style` - car body style (sedan or hachback)  
`drive-wheels` - which types of drive wheel (forward-fwd, reversed-rwd)  
`engine-location` - engine mounted location (front or back)  
`wheel-base` - расстояние между осями передних и задних колес  
`length` - car lenght  
`weight` - car weight  
`width` - car width  
`height` - car height  

In [4]:
df.shape

(205, 26)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 26 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   symboling          205 non-null    int64  
 1   normalized-losses  205 non-null    object 
 2   make               205 non-null    object 
 3   fuel-type          205 non-null    object 
 4   aspiration         205 non-null    object 
 5   num-of-doors       205 non-null    object 
 6   body-style         205 non-null    object 
 7   drive-wheels       205 non-null    object 
 8   engine-location    205 non-null    object 
 9   wheel-base         205 non-null    float64
 10  length             205 non-null    float64
 11  width              205 non-null    float64
 12  height             205 non-null    float64
 13  curb-weight        205 non-null    int64  
 14  engine-type        205 non-null    object 
 15  num-of-cylinders   205 non-null    object 
 16  engine-size        205 non

## Заполнение пропусков

Пропуски в этом датасете обозначены как `?`

In [6]:
for c in df.columns:
    print(c, len(df[df[c] == '?']))

symboling 0
normalized-losses 41
make 0
fuel-type 0
aspiration 0
num-of-doors 2
body-style 0
drive-wheels 0
engine-location 0
wheel-base 0
length 0
width 0
height 0
curb-weight 0
engine-type 0
num-of-cylinders 0
engine-size 0
fuel-system 0
bore 4
stroke 4
compression-ratio 0
horsepower 2
peak-rpm 2
city-mpg 0
highway-mpg 0
price 4


Удалите строки, для которых неизвестно значение price, так как это целевая переменная.

## Вопрос для Quiz

Сколько строк осталось в данных?

In [7]:
df = df[df['price'] != '?']
df.shape

(201, 26)

Заполните средним значением пропуски в столбцах для числовых признаков и самым популярным значением для категориальных признаков
* `num-of-doors`
* `bore`
* `stroke`
* `horsepower`
* `peak-rpm`

In [8]:
mode = df[df['num-of-doors'] != '?']['num-of-doors'].mode()[0]
df['num-of-doors'] = df['num-of-doors'].apply(lambda x: mode if x == '?' else x)

In [9]:
for col in ['bore', 'stroke', 'horsepower', 'peak-rpm']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    if col == 'peak-rpm':
        print(df[col].mean())
    df[col] = df[col].fillna(df[col].mean())

5117.587939698493


In [10]:
for c in df.columns:
    print(c, len(df[df[c] == '?']))

symboling 0
normalized-losses 37
make 0
fuel-type 0
aspiration 0
num-of-doors 0
body-style 0
drive-wheels 0
engine-location 0
wheel-base 0
length 0
width 0
height 0
curb-weight 0
engine-type 0
num-of-cylinders 0
engine-size 0
fuel-system 0
bore 0
stroke 0
compression-ratio 0
horsepower 0
peak-rpm 0
city-mpg 0
highway-mpg 0
price 0


## Вопрос для Quiz

Чему равно среднее значение `peak-rpm` до заполнения пропусков? Ответ округлите до целого числа.

Пропуски в столбце `normalized-losses` предскажите при помощи линейной регрессии по признакам
`symboling`, `wheel-base`, `length`, `width`, `height`, `curb-weight`, `engine-size`, `compression-ratio`, `city-mpg`, `highway-mpg` и заполните их предсказаниями

In [11]:
from sklearn.linear_model import LinearRegression

df_for_model = df[['symboling', 'wheel-base', 'length', 'width', 'height', 'curb-weight', 'engine-size', 'compression-ratio', 'city-mpg', 'highway-mpg', 'normalized-losses']]

X = df_for_model[df_for_model['normalized-losses'] != '?'].drop('normalized-losses', axis=1)
y = df_for_model[df_for_model['normalized-losses'] != '?']['normalized-losses']

X_test = df_for_model[df_for_model['normalized-losses'] == '?'].drop('normalized-losses', axis=1)

lr = LinearRegression()

lr.fit(X, y)

pred = lr.predict(X_test)

df.loc[df['normalized-losses'] == '?', 'normalized-losses'] = pred

## Вопрос для Quiz

Чему равно предсказание линейной регрессии на первом пропущенном значении? Ответ округлите до целого числа.

## 2. Кодирование категориальных признаков

1. Закодируйте бинарные признаки `fuel-type`, `aspiration`, `num-of-doors`, `engine-location` каждый отдельной колонкой, состоящей из 0 и 1.
Единицей кодируйте самую частую категорию.

In [12]:
for col in ['fuel-type', 'aspiration', 'num-of-doors', 'engine-location']:
    df[col] = (df[col] == df[col].mode()[0]).astype(int)

2. Вынесите в переменную `y` целевую переменную `price`, а все остальные колонки - в матрицу `X`.

Закодируйте признаки `make`, `body-style`, `engine-type`, `fuel-system` при помощи LeaveOneOutEncoder.

**Дальше все время работайте с объектами `X`, `y`.**

In [13]:
!pip install category-encoders==2.6.3
!pip install scikit-learn==1.3.2

In [14]:
X = df.drop('price', axis=1)
y = df['price']

In [15]:
from category_encoders.leave_one_out import LeaveOneOutEncoder

encoder = LeaveOneOutEncoder(cols=['make', 'body-style', 'engine-type', 'fuel-system'])

X = encoder.fit_transform(X, y)

In [16]:
X['body-style'].mean()

13207.129353233831

## Вопрос для Quiz

Чему равно среднее значение в столбце `body-style` после кодирования? Ответ округлите до целого числа.

3. Закодируйте признак `drive-wheels` при помощи OHE из библиотеки category_encoders.

In [17]:
from category_encoders.one_hot import OneHotEncoder

ohe = OneHotEncoder(cols=['drive-wheels'])

X = ohe.fit_transform(X)

In [18]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 201 entries, 0 to 204
Data columns (total 27 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   symboling          201 non-null    int64  
 1   normalized-losses  201 non-null    object 
 2   make               201 non-null    float64
 3   fuel-type          201 non-null    int64  
 4   aspiration         201 non-null    int64  
 5   num-of-doors       201 non-null    int64  
 6   body-style         201 non-null    float64
 7   drive-wheels_1     201 non-null    int64  
 8   drive-wheels_2     201 non-null    int64  
 9   drive-wheels_3     201 non-null    int64  
 10  engine-location    201 non-null    int64  
 11  wheel-base         201 non-null    float64
 12  length             201 non-null    float64
 13  width              201 non-null    float64
 14  height             201 non-null    float64
 15  curb-weight        201 non-null    int64  
 16  engine-type        201 non-null

4. В столбце `num-of-cylinders` категории упорядочены по смыслу. Закодируйте их подряд идущими числами, начиная с 1, согласно смыслу.

Подряд идущими числами означает - 1, 2, 3 и так далее без пропусков.

In [19]:
X['num-of-cylinders'].value_counts()

,count
num-of-cylinders,
four,157
six,24
five,10
two,4
eight,4
three,1
twelve,1


In [20]:
mapping = {
    'two': 1,
    'three': 2,
    'four': 3,
    'five': 4,
    'six': 5,
    'eight': 6,
    'twelve': 7
}

X['num-of-cylinders'] = X['num-of-cylinders'].map(mapping)

In [21]:
X.shape

(201, 27)

## Вопрос для Quiz

Сколько столбцов получилось в матрице `X`?

In [22]:
X['normalized-losses'] = X['normalized-losses'].astype(float)
X['bore'] = X['bore'].astype(float)
X['stroke'] = X['stroke'].astype(float)
X['horsepower'] = X['horsepower'].astype(float)
X['peak-rpm'] = X['peak-rpm'].astype(float)

y = y.astype(float)

Разбейте данные на тренировочную и тестовую часть в пропорции 3 к 1, зафиксируйте random_state = 42.

In [23]:
from sklearn.model_selection import train_test_split

Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)

Масштабируйте данные при помощи MinMaxScaler.

Обучайте масштабирование на тренировочных данных, а потом примените и к трейну, и к тесту.

In [24]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

Xtrain = pd.DataFrame(scaler.fit_transform(Xtrain), columns=Xtrain.columns)
Xtest = pd.DataFrame(scaler.transform(Xtest), columns=Xtest.columns)

Обучите на тренировочных данных линейную регрессию, сделайте предсказание на тесте и вычислите значение $R^2$ на тестовых данных.

In [25]:
from sklearn.metrics import r2_score

lr = LinearRegression()

lr.fit(Xtrain, ytrain)
pred = lr.predict(Xtest)

r2_score(ytest, pred)

0.9088107324676914

## Вопрос для Quiz

Чему равно значение $R^2$ на тестовых данных? Ответ округлите до сотых.